# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² mental health survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"License: {metadata['license']}")
print(f"Version: {metadata['version']}")
print(f"Published at: {metadata['datePublished']}")
print("\nAuthors (referenced by @id):")
for author in metadata['author']:
    print(author['@id'])

# For future reference, let's save some IDs and list them
dataset_id = metadata['@id']

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list all available record sets and their fields as referenced by their `@id` values.

In [ ]:
# Get record sets from the dataset metadata
record_sets = metadata.get('recordSet', [])
if not record_sets:
    print('No record sets found in the dataset metadata.')
else:
    print('Record sets (@id):')
    for rs in record_sets:
        # Each record set might be a dict with @id
        rs_id = rs.get('@id') if isinstance(rs, dict) else rs
        print(f"- {rs_id}")
        try:
            rs_obj = dataset.metadata._record_sets_dict[rs_id]
            field_ids = [field['@id'] for field in rs_obj.get('field', []) if isinstance(field, dict) and '@id' in field]
            print(f"  Fields (@id): {field_ids}")
        except Exception:
            print("  (Unable to retrieve fields for this record set @id)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use record set and field `@id` values as retrieved above.

We'll load all available record sets (if any) and display the columns for one.

In [ ]:
# Prepare list of record set @ids
record_set_ids = []
for rs in record_sets:
    if isinstance(rs, dict) and rs.get('@id'):
        record_set_ids.append(rs['@id'])
    elif isinstance(rs, str):
        record_set_ids.append(rs)
dataframes = {}

if not record_set_ids:
    print("No record sets available for extraction.")
else:
    # Extract data from each record set
    for record_set in record_set_ids:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set @id: {record_set} - Shape: {dataframes[record_set].shape}")

    # Display columns for the first record set if present
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns (@id) in DataFrame for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())

    print(f"\nPreview records from {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping data by attributes.

We will operate on specific fields referenced by their `@id`. For demonstration, let's assume there is a numeric field with `@id = 'gad_7_score'` (Generalized Anxiety Disorder, GAD-7 score) and a demographic field `@id = 'level_of_education'`.

In [ ]:
# Choose main record set and field IDs
record_set_id = record_set_ids[0] if record_set_ids else None
# Example numeric and group fields, update as per dataset actual columns
numeric_field = 'gad_7_score'     # GAD-7 score field @id
group_field = 'level_of_education'  # Education field @id

if record_set_id and numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped average {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in DataFrame columns for record set {record_set_id}. Available columns:")
    print(dataframes[record_set_id].columns.tolist() if record_set_id else [])

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

Here, for illustration, we plot the distribution of GAD-7 scores and their relationship to education level.

In [ ]:
if record_set_id and numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field], bins=15, kde=True, color='skyblue')
    plt.title('Distribution of GAD-7 Scores (@id: gad_7_score)')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

    if group_field in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'GAD-7 Scores by Education Level (@id: {group_field})')
        plt.xlabel('Level of Education')
        plt.ylabel('GAD-7 Score')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization could not be generated due to missing columns.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we have loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. We examined the structure of the dataset by referencing record sets and fields via their `@id` values, loaded data into pandas DataFrames for analysis, filtered and normalized numeric scores, and grouped data by demographic factors for further insights.

The visualizations demonstrated how GAD-7 anxiety scores vary across education levels, and a similar approach can be applied to PHQ-9, PSQ, or other available indicators. For precise analysis, reference actual field and column `@id` values as listed in the Croissant schema.

This workflow illustrates the reproducibility and AI-readiness enabled by the Croissant schema standard and the `mlcroissant` library, supporting FAIR data practices in mental health research.